# 01 Data collection

The collector lives in `src/collect.py` so that the same code runs from `make collect`,
from this notebook and from a rerun after an interruption. This notebook checks what
the collection produced rather than repeating its logic.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, duckdb
pd.set_option('display.width', 160)

In [ ]:
from src.config import API_BASE, AREA_CODES, PAGE_LIMIT
print('base path:', API_BASE)
print('page limit:', PAGE_LIMIT)
print('area codes:', AREA_CODES)

## The sweep is a partition

`area` is not in the project payload, so each project is fetched under exactly one
area code. If the 8 counts sum to the reported total, nothing was missed or duplicated.

In [ ]:
import gzip, json, collections
from pathlib import Path
pages = sorted(Path('../data/raw/projetos').rglob('*.json.gz'))
print(f'{len(pages)} raw pages on disk')
per_area = collections.Counter()
totals = {}
for p in pages:
    blob = json.load(gzip.open(p, 'rt', encoding='utf-8'))
    meta = blob['_meta']
    per_area[meta['area_code']] += len(blob['response']['_embedded']['projetos'])
    totals[meta['area_code']] = meta['total_at_collection']
for code_, n in sorted(per_area.items()):
    print(f"area {code_}: collected {n:,} of {totals[code_]:,} reported")
print('sum collected:', f'{sum(per_area.values()):,}')

## Every page must read back

A file that exists is not a page that parses. One page of 618 was found corrupt
during development, so this check is part of the record.

In [ ]:
bad = []
for p in pages:
    try:
        json.load(gzip.open(p, 'rt', encoding='utf-8'))
    except Exception as exc:
        bad.append((p.name, type(exc).__name__))
print('unreadable pages:', bad or 'none')